# Writing client code: notebooks and apps

This notebook is for whoever writes code against `ess.dispatch`: a notebook, a local app, a service-backed app, or a program.
It follows one thread, from a for loop over runs to a sum of runs that grows as they arrive.
Every step shows what the framework guarantees, and what the client must do in return.

The workflows are the toy SANS reduction of `toy_sans.py`.
A run's file holds counts per pixel and a monitor count, and a sample run's file the sample's transmission.
`IOFQ` reduces one sample run and one can run to an I(Q) curve, `IOFQ_MULTI` sums several of each, and `FORWARD` reads a curve and returns its first point.
[workflow-authors.ipynb](workflow-authors.ipynb) shows how these specs and their bindings are written.
Here they are given, as an installed package would give them.

A spec has *params*, configuration that a request gives by value, and *inputs*, data that a request gives by *reference*: a dataset, or an output of a record.

| Spec | Params | Inputs | Outputs |
|---|---|---|---|
| `IOFQ` | `reduction: Reduction` (`bins=2`), `transmission: Transmission \| None` | `sample: SampleFile`, `can: CanFile` | `iofq: IofQ` |
| `IOFQ_MULTI` | `reduction: Reduction`, `transmission: list[Transmission \| None]` | `sample: list[SampleFile]`, `can: list[CanFile]` | `iofq: IofQ` |
| `FORWARD` | (none) | `curve: IofQ` | `i0: ForwardIntensity` |

`SampleFile` and `CanFile` are files; `IofQ` and `ForwardIntensity` are scipp variables.
`IOFQ_MULTI` is made from `IOFQ` by `tabulate`, with a table of sample runs, each with its transmission, and a table of can runs (section 6).

In [ ]:
import tempfile
from pathlib import Path

from ess.dispatch import Backend, Client, Request, SubmitError, Template, dataset
from ess.dispatch.testing import FakeDatasets, FakeStore
from toy_sans import (
    BINDINGS,
    CAN_RUNS,
    FORWARD,
    IOFQ,
    IOFQ_MULTI,
    SAMPLE_RUNS,
    IofQParams,
    Reduction,
    Transmission,
)

## Setup

The framework reads raw data through a *dataset source*.
Here a fake one writes a few runs of proposal `p1` as files in a temporary folder, and a fake store stands in for disk.

In [ ]:
folder = tempfile.TemporaryDirectory()
datasets = FakeDatasets(proposal='p1', directory=Path(folder.name))
for run, data in SAMPLE_RUNS.items():
    datasets.measure(run, data, role='sample', temperature=f'{run + 149}K')
for run, data in CAN_RUNS.items():
    datasets.measure(run, data, role='can')

backend = Backend(datasets, BINDINGS, store=FakeStore())
client = Client(backend, proposal='p1', submitter='alice')

`local(proposal='p1', datasets=datasets, bind=BINDINGS, store=...)` does the same in one call.
This notebook keeps `backend` to open a second client later, as another process would with `connect(url, proposal='p1')` on the service.
The service is not implemented; everything below runs in this process.

## 1. From a for loop to records

A notebook without the framework calls the reduction directly:

```python
curves = {}
for run in (101, 102, 103):
    curves[run] = reduce_iofq(sample=run, can=110, bins=2)
```

With the framework, each call becomes a request: a spec, its params, and its inputs.
`dataset(run=101)` names a dataset; every parameter has a default here, so the params are left out.
`submit` returns a *record* at once, and the requests run in parallel.

In [ ]:
can = dataset(run=110)
records = {}
for run in (101, 102, 103):
    records[run] = client.submit(IOFQ, inputs={'sample': dataset(run=run), 'can': can})
client.wait(records)
client.output(records[101], 'iofq')

A record holds the request as the framework accepted it: every parameter filled in, defaults included, and each dataset resolved from the name the user gave (`run:101`) to its identity.
It holds no output values.
A record never changes; only its status moves from `pending` to `completed`, `failed`, or `cancelled`.

In [ ]:
record = records[101]
record.request.params, record.request.inputs, client.status(record)

Every call takes one record, a list, or a dict, and returns the same shape:

In [ ]:
client.status(records)

**A request that cannot run is refused at submission**, with an error that names the field.
If one request of a call is refused, none is submitted, so a batch with one typo starts nothing.

In [ ]:
batch = {
    'good': Request(IOFQ, inputs={'sample': dataset(run=102), 'can': can}),
    'typo': Request(
        IOFQ,
        params={'reduction': {'bins': 0}},
        inputs={'sample': dataset(run=102), 'can': can},
    ),
}
try:
    client.submit(batch)
except SubmitError as error:
    print(error)

Params may be given as a dict of the params model's fields, as above, or as an instance of the model, which a type checker checks (section 2); both make the same request.
`Request(spec, params=..., inputs=...)` makes a request without submitting it, so that several go into one call.

An input is always a reference.
A file too is named through the dataset source, by run, by PID, or by `dataset(path=...)`, so that every file a request reads has an identity in the record; a path given as a string is refused:

In [ ]:
try:
    client.submit(IOFQ, inputs={'sample': '/data/run-102.json', 'can': can})
except SubmitError as error:
    print(error)

## 2. Chains: a reference to a pending record

An input may also be an output of a record, given as `record.ref(output)`.
The record may still be pending, so a chain is submitted without waiting.
The framework runs `FORWARD` once the I(Q) has completed.

In [ ]:
curve = client.submit(
    IOFQ,
    params=IofQParams(reduction=Reduction(bins=4)),
    inputs={'sample': dataset(run=103), 'can': can},
)
i0 = client.submit(FORWARD, inputs={'curve': curve.ref('iofq')})
client.output(i0, 'i0')

`client.output` waits for the record.
The framework checks at submission that the output *fits* the input, by their types: an `IofQ` output fits a field typed `IofQ`, or `sc.Variable`, of which `IofQ` is a `NewType`.
`ForwardIntensity`, another `NewType` of `sc.Variable`, does not fit `IofQ`:

In [ ]:
try:
    client.submit(FORWARD, inputs={'curve': i0.ref('i0')})
except SubmitError as error:
    print(error)

## 3. Who keeps a value

An output value, such as the I(Q) array, lives while something keeps it, and nothing else keeps it:

| Keeper | Keeps the value until |
|---|---|
| the client that submitted the record | the client releases the record, or ends |
| a pending request that reads it | that request's workflow has returned |
| a stage, an accumulator's template, or a push that references it | the binding has taken it, or the stage, accumulator, or push is released or stopped |
| a persist request | the value is written |
| the store, once the value is written | the proposal's history is dropped |

A record object, a reference, or a label keeps nothing.
Releasing drops the client's hold; the record stays, but its value is gone for good:

In [ ]:
client.release(curve)
try:
    client.output(curve, 'iofq')
except LookupError as error:
    print(error)

`FORWARD` had read it already, so `i0` is unaffected.
Had `FORWARD` still been pending, it would have kept the curve until it ran.

Two consequences that a client author must know:

- **Work that nothing keeps is cancelled.** Releasing a pending record that no pending request reads cancels it, and closing a client releases everything it keeps. A notebook that submits a batch and closes its client before the batch has run cancels the batch, unless it was persisted.
- **Another client never reads the values this client keeps**, not even in the same process. It reads only persisted values.

On the service, the memory a client keeps is capped, and a client that makes no call for a while is ended (its *lease*).
A long-lived app therefore releases what it no longer shows.

## 4. Persist, and find it the next day

`persist=` at submission makes the store the only keeper.
This is how an unattended batch is submitted: the client keeps nothing, and the records complete once their outputs are written.
A *label* names the batch, and a dict's keys become the *members*:

In [ ]:
night = client.submit(
    {
        f'{run + 149}K': Request(IOFQ, inputs={'sample': dataset(run=run), 'can': can})
        for run in (101, 102, 103)
    },
    label='scan',
    persist=True,
)
client.wait(night)

The next morning, another client of the same proposal finds the batch by its label and reads from the store.
Here a second client in this process stands in for it:

In [ ]:
morning = Client(backend, proposal='p1', submitter='bob')
latest = morning.latest('scan', member='250K')
morning.output(latest, 'iofq')

`client.persist(record)` instead adds the store next to the client, for a record the client keeps.
On the service, records are the proposal's history: they last until the proposal has been idle for days to weeks, and are then dropped as a whole.
With `local()`, history is held in memory and ends with the process.
What must last longer is published to SciCat (not implemented).

## 5. Templates: tune with a stage

A *template* is a spec with some params and inputs fixed, and *blanks*, the names of the parameters and inputs left open.
It takes the fixed values as a request does, as `params=` and `inputs=`.
A *stage* is a template the backend keeps for this client, so that what depends only on the fixed values is computed once.
Tuning `bins` then does not load the runs again:

In [ ]:
tune = client.stage(
    Template(
        IOFQ, inputs={'sample': dataset(run=101), 'can': can}, blanks=('reduction',)
    )
)
for bins in (1, 2, 4):
    tuned = client.compute(tune, params={'reduction': Reduction(bins=bins)})
    print(bins, client.output(tuned, 'iofq').values)

A call through a stage makes the record of the plain request, the one with every value given.
Nothing in the record says it went through a stage, and its outputs are those of the plain request (the *caching* symmetry):

In [ ]:
tuned.request

A stage needs a binding that makes stages, such as one built from a sciline pipeline, which computes the fixed part once.
A stage over any other binding is refused when it is made:

In [ ]:
try:
    client.stage(Template(FORWARD, blanks=('curve',)))
except SubmitError as error:
    print(error)

## 6. Sum runs as they arrive: an accumulator

A *table* is a set of list fields of equal length.
Its *index* is the field that names it, such as `sample`; its other *columns* are attached to the index, such as `transmission`, one per sample run.
Each position is a *row*.
`IOFQ_MULTI` has two tables: `sample` with `transmission`, and `can` alone ([README, Combining runs](../README.md#combining-runs-tables)).

An *accumulator* is a template whose blanks are tables, named by their indexes, kept by the backend for this client.
Each *push* gives rows of one table: its index and the columns attached to it, by name, as lists of equal length.
State n, the accumulator after n pushes, has the outputs of the plain request whose tables hold the rows of the first n pushes.

In [ ]:
total = client.accumulator(Template(IOFQ_MULTI, blanks=('sample', 'can')))
total.push(can=[can])  # state 1
total.push(sample=[dataset(run=101)])  # state 2
state_2 = client.output(total, 'iofq')
total.push(sample=[dataset(run=102)], transmission=[Transmission(value=0.9)])  # state 3
state_3 = client.output(total, 'iofq')
state_2.values, state_3.values

`push` returns once the backend has logged it; the backend adds it later.
A read *pins* the state after the pushes logged so far and waits for it.
State 1 has no sample row, so a read of it would be refused: `sample` is a required index, and its table needs a row.

A value that differs per run is in the run's file, and a value that the file lacks or has wrong is a column next to the run.
Run 102 comes with a transmission, which replaces the one in its file; run 101 takes the one in its file, since a column left out of a push is `None` in each row.

State 3 equals the plain request over the same rows (the *arrival* symmetry).
`tabulate` makes the params model of `IOFQ_MULTI` without a static type, so its params are given as a dict:

In [ ]:
plain = client.compute(
    IOFQ_MULTI,
    params={'transmission': [None, Transmission(value=0.9)]},
    inputs={'sample': [dataset(run=101), dataset(run=102)], 'can': [can]},
)
client.output(plain, 'iofq').values

With one row in each table, a request of `IOFQ_MULTI` gives the outputs of the request of `IOFQ` with the same values (the *one-row* symmetry).

Behind the accumulator, the binding of `IOFQ_MULTI` makes a *driver*, which the backend keeps for the accumulator.
The driver holds the sums of numerators and denominators, and a push loads its runs and adds them.
Only a spec whose binding makes drivers has accumulators; the backend refuses any other when the accumulator is made.

Rows of different tables have nothing to do with each other, so a push names one table, through its index.
Values that belong to one run, such as its transmission, are columns of its row.
A push may give several rows of its table, which enter one state; a push of no rows is refused.

**Readers never see a value change.**
`client.output` of an accumulator returns a copy, so `state_2` above still holds state 2.
A request may read a state too, through `total.ref(...)`.
It reads the output of the driver without a copy, so the backend holds back the next push until that request's workflow has returned:

In [ ]:
forward_3 = client.submit(FORWARD, inputs={'curve': total.ref('iofq')})  # pins state 3
total.push(sample=[dataset(run=103)])
total.push(can=[dataset(run=111)])
client.output(forward_3, 'i0').value, state_3['Q', 0].value

The order of the calls is the order of the states, as if each call ran after the one before it, although pushes and reads run in parallel.

The index values of a table are distinct, as in a pandas DataFrame.
A run pushed again is refused, since pushing a run twice is usually a cell run twice; so is a column pushed without its index:

In [ ]:
try:
    total.push(sample=[dataset(run=103)])
except SubmitError as error:
    print(error)
try:
    total.push(transmission=[Transmission(value=0.5)])
except SubmitError as error:
    print(error)

At the end, `freeze` turns the last state into an ordinary record, without a copy, and the accumulator takes no more pushes:

In [ ]:
summed = client.freeze(total)
client.output(summed, 'iofq').values

## 7. Where a result came from

Every record can say which requests and datasets produced it.
The record of a frozen state holds the plain request over its rows, with every column, and does not name the accumulator:

In [ ]:
provenance = client.provenance(summed)
summed.request.params, [str(d) for d in provenance.datasets()]

## 8. Flavours of client code

Everything above is the same API for every client.
What differs between flavours is which calls they rely on, and what they must take care of.

| Flavour | Makes its client with | Typically | Must take care of |
|---|---|---|---|
| notebook | `local(...)` | `submit`, `output`, stages, accumulators | releasing large values; not modifying values that `output` returned (see below) |
| local app (not a notebook) | `local(...)` | the same, behind a form | releasing what it no longer shows; choosing the rows of each table |
| app against the service | `connect(url, proposal=...)` (not implemented) | `submit(..., persist=True)`, labels, `latest` | the memory cap and the lease; reading results from the store |
| batch or automatic reduction | either | `apply`, `Rule`, `TriggerLoop` | choosing what to persist |
| program, CLI, agent | `connect(...)` (not implemented) | `SpecId`, `SerializedWorkflowSpec` | importing no workflow package |

Points that hold for apps but not for notebooks:

- **Forms come from the spec.** `IOFQ.serialize()` gives the JSON Schema of the params, for a form, and the type of data each input takes, such as `SampleFile`, for a picker of datasets or records. An app that imports only a package's specs, kept in a module apart from the workflow code, builds forms without the workflow's dependencies ([ADR 0007](../adr/0007-packages-split-by-dependencies.md)).
- **Errors name the field at fault**, in the message of a `SubmitError` (`typo: toy-sans-iofq/v1: reduction.bins: ...` above). The error carries no structured field path, so a form that shows the error next to its field must read it from the message.
- **An app that does not import the workflow package** names the spec by `SpecId(name='toy-sans-iofq', version=1)` and gets the same checks.
- **The framework knows no technique.** Which sample run goes with which can run, and which runs form the rows of a table, is the app's decision.

**Batch reduction** fills a template once per dataset with `apply`: each dataset fills the one blank input, here `sample`.
`apply` submits nothing and returns the requests keyed by a metadata field:

In [ ]:
from ess.dispatch import Selector, apply

samples = client.datasets.list(Selector(role='sample'))
requests = apply(
    Template(IOFQ, inputs={'can': can}, blanks=('sample',)),
    samples,
    client.datasets,
    member_field='temperature',
)
batch = client.submit(requests, label='batch', persist=True)
client.wait(batch)
sorted(r.member for r in client.records(label='batch'))

A `Rule` holds the same as plain data (a template, a selector, a label), and the `TriggerLoop` submits it for each new dataset; see [automatic-reduction.md](../automatic-reduction.md).

**Values in the user's process are not copied.**
In a notebook, `client.output` of a record returns the value the framework holds, not a copy.
Modifying it in place changes what later requests read and what `persist` writes.
This is open question 7 of the [README](../README.md); until it is settled, treat outputs as read-only.

## Summary: what client code relies on, and what it owes

| Relies on | Owes |
|---|---|
| a record never changes, only its status | keeping (or persisting) what it still needs, and releasing the rest |
| a refused call submits nothing, and names the field | treating output values as read-only |
| a value lives while something keeps it, and only then | giving each input by reference, a file through the dataset source |
| a request runs once the records it references have completed | choosing the rows of each table: the framework knows no technique |
| a reference is checked by type at submission, and the workflow gets the data it names | |
| a reader never sees a value change | |
| a call through a stage, or a state of an accumulator, gives the outputs of its plain request | |
| where a request runs changes neither its record nor its outputs | |

[framework-promises.ipynb](framework-promises.ipynb) shows what keeping these promises takes, in this process and on the service.

In [ ]:
client.close()
morning.close()
backend.close()
folder.cleanup()